# 02. Advanced Preprocessing, Scaler Benchmarking & Leakage-Free Splitting
**Project**: Solar & Electrical Power Fault and Anomaly Detection Using Stacking Ensemble Learning
**Objective**: Compare all major feature scalers (StandardScaler, RobustScaler, MinMaxScaler, QuantileTransformer, PowerTransformer) to select the optimal transformation pipeline for distance/kernel and tree-based models.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, MinMaxScaler, QuantileTransformer, PowerTransformer
)
from src.config import (
    STATION_FILES, PROCESSED_DATA_DIR, MODELS_DIR, FEATURE_COLS, 
    CATEGORY_LABELS, SEVERITY_LABELS, RAW_SEVERITY_MAP, RANDOM_STATE
)
from src.preprocessing import build_curated_benchmark_dataset, create_leakage_free_splits
from src.feature_engineering import add_electrical_features

print('All preprocessing modules and scalers imported successfully!')

## 1. Curate Multi-Station Benchmark Data
- Preserves 100% of all 30,142 controlled fault samples (zero rare event loss).
- Forms contiguous 300-second (5-minute) normal operating temporal blocks to eliminate cross-second leakage.

In [ ]:
df_curated = build_curated_benchmark_dataset(normal_block_length_s=300, max_normal_blocks_per_st=50)
print('Curated dataset shape:', df_curated.shape)
df_curated[['timestamp', 'station_id', 'voltage_V', 'current_A', 'power_W', 'fault_category', 'severity']].head(5)

## 2. Leakage-Free Stratified Group Splitting
Splits the dataset using `StratifiedGroupKFold` on `group_id` so complete fault episodes and normal time windows are kept intact within single partitions.

In [ ]:
train_df, test_df = create_leakage_free_splits(df_curated)
print('Train partition rows:', len(train_df))
print('Test partition rows: ', len(test_df))

# Automated verification of zero event leakage
train_events = set(train_df[train_df['event_id'] != 0]['group_id'].unique())
test_events = set(test_df[test_df['event_id'] != 0]['group_id'].unique())
overlap = train_events.intersection(test_events)
print('Overlapping fault events (MUST BE 0):', len(overlap))
assert len(overlap) == 0, 'Data leakage detected across train and test!'

## 3. Comparing 5 Different Scalers on Electrical Sensor Features
We compare:
1. **StandardScaler**: Standardizes by removing mean and scaling to unit variance.
2. **RobustScaler**: Uses median and Interquartile Range (IQR); highly resilient to extreme electrical spikes/sags.
3. **MinMaxScaler**: Rescales all feature dimensions into $[0, 1]$.
4. **QuantileTransformer**: Maps feature distributions to uniform/Gaussian, damping non-linear skewness.
5. **PowerTransformer (Yeo-Johnson)**: Parametric power transform that stabilizes variance and enforces normality.

In [ ]:
scalers = {
    'StandardScaler': StandardScaler(),
    'RobustScaler': RobustScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'QuantileTransformer': QuantileTransformer(n_quantiles=1000, random_state=RANDOM_STATE),
    'PowerTransformer': PowerTransformer(method='yeo-johnson')
}

X_train = train_df[FEATURE_COLS].copy()
feature_to_plot = 'current_A'

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Raw distribution
sns.histplot(X_train[feature_to_plot], kde=True, ax=axes[0], color='gray', bins=40)
axes[0].set_title('Raw ' + feature_to_plot)

# Visualizing transformed distributions
for idx, (name, scaler) in enumerate(scalers.items(), start=1):
    transformed = scaler.fit_transform(X_train[[feature_to_plot]])
    sns.histplot(transformed, kde=True, ax=axes[idx], color='teal', bins=40)
    axes[idx].set_title(name)

plt.tight_layout()
plt.show()

## 4. Benchmark Scalers on Downstream Classifier (KNN & Linear SVM)
Measures which scaler produces the highest Macro-F1 score on the held-out test set.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

X_tr_raw = train_df[FEATURE_COLS].values
y_tr = train_df['target_category_idx'].values
X_te_raw = test_df[FEATURE_COLS].values
y_te = test_df['target_category_idx'].values

results = []
for name, scaler in scalers.items():
    # Fit strictly on training split to eliminate data leakage
    X_tr_s = scaler.fit_transform(X_tr_raw)
    X_te_s = scaler.transform(X_te_raw)
    
    knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    knn.fit(X_tr_s, y_tr)
    preds = knn.predict(X_te_s)
    
    score = f1_score(y_te, preds, average='macro', zero_division=0)
    results.append({'Scaler': name, 'Macro-F1 (KNN %)': round(score * 100, 2)})

benchmark_df = pd.DataFrame(results).sort_values(by='Macro-F1 (KNN %)', ascending=False)
display(benchmark_df)

## 5. Export Best Scaler & Processed Partitions
Saves the optimal preprocessor to `models/preprocessors/` and verifies saved dataset files.

In [ ]:
best_scaler_name = benchmark_df.iloc[0]['Scaler']
print('Highest performing scaler:', best_scaler_name)

best_scaler = scalers[best_scaler_name]
best_scaler.fit(X_tr_raw)
save_path = MODELS_DIR / 'preprocessors' / 'scaler.pkl'
joblib.dump(best_scaler, save_path)
print('Saved fitted optimal scaler to:', save_path)